In [1]:
import dash
from dash import html,set_props,Input,Output,State
from feffery_dash_utils.style_utils import style
import feffery_antd_components as fac
import feffery_utils_components as fuc
import feffery_antd_charts as fact
import pandas as pd
import colorlover as cl
import os

In [2]:
# 加载数据
if os.path.exists(r'./datas/stock-ticker.csv'):
    df = pd.read_csv(r'./datas/stock-ticker.csv')
    df['Date'] = pd.to_datetime(df['Date']).dt.strftime('%Y-%m-%d')
else:
    df = pd.DataFrame(columns=['Date', 'Stock', 'Open', 'High', 'Low', 'Close', 'Volume', 'ExDividend', 'SplitRatio', 'AdjOpen', 'AdjHigh', 'AdjLow', 'AdjClose', 'AdjVolume', 'Stock'])

# 加载股票列表
if os.path.exists(r'./datas/tickers.csv'):
    df_tickers = pd.read_csv(r'./datas/tickers.csv')
    # 构造字典(label为显示值，value为实际值)
    ticker_options = [
            {'label':f"{row['Symbol']} - {row['Company']}",'value': row['Symbol']}
            for _,row in df_tickers.iterrows() # 获取索引和行内容
        ]
else:
    ticker_options = [{'label': s,'value': s} for s in df.Stock.unique()]

# 获取一套对比强烈的配色方案，用于区分不同股票的布林带
colorscale = cl.scales['9']['qual']['Paired']

In [3]:
# 页面设计
app = dash.Dash(
    __name__
)

app.layout = html.Div(
    [
        fuc.FefferyTopProgress(
            html.Div(
                [
                    # logo标题 & 筛选
                    fac.AntdRow(
                        # 整体分一列
                        fac.AntdCol(
                            [
                                # 全局消息提示
                                fac.Fragment(id="global-message"),
                                # logo & 标题 & 标题说明
                                fac.AntdSpace(
                                    [
                                        
                                        # logo & 标题
                                        fac.AntdSpace(
                                            [
                                                # logo
                                                html.Img(src="/assets/imgs/Stocks.svg", height=48),
                                                # 标题
                                                fac.AntdText("金融资产走势分析",style=style(fontSize=26))
                                            ]
                                        ),
                                        # 标题说明
                                        fac.AntdText(
                                            "用于测试的假数据，只用于功能开发与测试",
                                            type="secondary",
                                        ),
                                    ],
                                    direction="vertical", # 垂直布局
                                    size=0
                                ),
                                # 分割线
                                fac.AntdDivider(),
                                # 筛选条件
                                fac.AntdRow(
                                    # 条件筛选 & 查询按钮
                                    [
                                        # 条件筛选
                                        fac.AntdCol(
                                            # 构建查询组表单项(含标签)
                                            fac.AntdFormItem(
                                                # 下拉框组件
                                                fac.AntdSelect(
                                                    id="stock-ticker-input",
                                                    placeholder="请选择",
                                                    # options=ticker_options,
                                                    options=df['Stock'].unique().tolist(),
                                                    # mode="multiple",
                                                    allowClear=True,
                                                    maxTagCount="responsive",
                                                ),
                                                label="股票代码",
                                                layout="horizontal", # horizontal水平/vertical垂直
                                            ),
                                            span=20
                                        ),
                                        # 查询按钮
                                        fac.AntdCol(
                                            fac.AntdButton(
                                                "查询分析",
                                                id="stock-ticker-button",
                                                type="primary",
                                                block=True, # 按钮设置为"块级"，自动铺满
                                                loadingChildren="分析中" # 回调后显示"分析中"
                                            ),
                                            span=4
                                        )
                                    ],
                                    gutter=[25,15] # 左右间距25，上下间距15
                                )
                            ],
                            span=24,
                            className="my-card"
                        ),
                        style = {"height":"22%","marginBottom":"10px"}
                    ),

                    # 数据展示
                    fac.AntdRow(
                        [
                            # 整体分三行
                            fac.AntdCol(
                                html.Div(
                                    [
                                        fac.AntdRadioGroup(
                                            id="chart-type-switch",
                                            options=[
                                                {"label": "K线图", "value": "kline"},
                                                {"label": "布林带图", "value": "bollinger"}
                                            ],
                                            value="kline",
                                            optionType="button"
                                        )
                                    ],
                                    className="my-card"
                                ),
                                span=24,
                                style = {"height":"10%","marginBottom": "-8px"}
                            ),
                            fac.AntdCol(
                                html.Div(
                                    [],
                                    id="graphs-container",
                                    className="my-card"
                                ),
                                span=24,
                                style = {"height":"45%","marginBottom": "10px"}   
                            ),
                            fac.AntdCol(
                                html.Div(
                                    [],
                                    id="table-container",
                                    className="my-card"
                                ),
                                span=24,
                                style = {"height":"45%","marginBottom": "10px"}
                            ),
                        ],
                        style = {"height":"calc(78% - 10px)"}
                    )
                ],
                style={"height": "100%"}
            ),
            style={"height":"100%"},
            listenPropsMode="include",
            includeProps=['graphs-container','table-container'],
            minimum=0.4
        )
    ],
    className="main-container"
)

# 计算金融分析中非常经典的指标——布林带
# price: 通常是一个包含股票收盘价（Close Price）
# window=10: 时间窗口（周期）。这里默认计算过去10个交易日的数据
# stdev=5: 标准差倍数。它决定了上下轨距离中轨的远近（通常金融领域常用 2 倍，你的代码中设为了 5 倍）
def calculate_bbands(price,window=10,stdev=5):
    """计算布林带技术指标"""
    rolling_mean = price.rolling(window=window).mean()
    rolling_std = price.rolling(window=window).std()
    return (
        rolling_mean,
        rolling_mean + (rolling_std * stdev),
        rolling_mean - (rolling_std * stdev)
    )


@app.callback(
    output=dict(
        graphs_container = Output("graphs-container","children"),
        table_container = Output("table-container","children")
    ),
    inputs=dict(
        nClicks=Input("stock-ticker-button","nClicks"),
        chart_type=Input("chart-type-switch", "value")
    ),
    state=dict(
        stock_ticker = State("stock-ticker-input","value")
    ),
    running=[(Output("stock-ticker-button","loading"),True,False)],
    prevent_initial_call=True
)
def update_stock_analysis(nClicks,stock_ticker,chart_type):
    if not stock_ticker:
        # 给出系统提示
        set_props(
            "global-message",
            {"children": fac.AntdMessage(content="请先完善查询条件", type="warning")}
        )
        return dict(
            graphs_container=dash.no_update,
            table_container=dash.no_update
        )

    # 数据处理，过滤出股票数据并排序
    dff = df[df['Stock'] == stock_ticker].sort_values('Date')

    if dff.empty:
        return dict(
            graphs_container=fac.AntdEmpty(description="未找到相关数据"),
            table_container=fac.AntdEmpty(description="未找到相关数据")
        )

    # 计算布林带指标
    bb_mean, bb_upper, bb_lower = calculate_bbands(dff['Close'])
    # 创建布林带图数据
    plot_data = []
    for idx, row in dff.iterrows():
        # 收盘价始终添加
        plot_data.append({'date': row['Date'], 'value': row['Close'], 'type': '收盘价'})
        # 使用相同的索引访问布林带值
        if pd.notna(bb_mean[row.name]):
            plot_data.append({'date': row['Date'], 'value': bb_mean[row.name], 'type': '中轨'})
            plot_data.append({'date': row['Date'], 'value': bb_upper[row.name], 'type': '上轨'})
            plot_data.append({'date': row['Date'], 'value': bb_lower[row.name], 'type': '下轨'})

    
    chart_data = dff.rename(columns={
        'Date': 'date',
        'Open': 'open',
        'Close': 'close',
        'High': 'high',
        'Low': 'low'
    }).to_dict('records')

    
    # 构建 Antd 表格数据
    table_data = dff.to_dict('records')

    # 图表
    if chart_type == "kline":
        chart = fact.AntdStock(
            data=chart_data,
            xField='date',
            yField=['open', 'close', 'high', 'low'],
            # 这里的配色符合中国股市习惯：红涨绿跌
            fallingFill='#26a69a',
            risingFill='#ef5350',
            stockStyle={
                'stroke': '#666',
                'lineWidth': 0.5,
            },
            tooltip={
                'crosshairs': {
                    'type': 'xy',
                    'textBackground': {
                        'padding': [4, 8],
                        'borderRadius': 4,
                        'fill': '#333',
                    },
                },
            },
            # 添加缩略轴，方便查看长周期数据
            slider={
                'start': 0.5,
                'end': 1,
            },
            style={'height': '100%'}
        )
    else:
        chart = fact.AntdLine(
            data=plot_data,
            xField='date',
            yField='value',
            seriesField='type',
            color=['#1890ff', '#666', '#ccc', '#ccc'],
            smooth=True,
            height=300
        )

    # 表格
    table_container = fac.AntdTable(
        columns=[
            {"title": "日期", "dataIndex": "Date"},
            {"title": "开盘价", "dataIndex": "Open"},
            {"title": "最高价", "dataIndex": "High"},
            {"title": "最低价", "dataIndex": "Low"},
            {"title": "收盘价", "dataIndex": "Close"},
        ],
        data=table_data,
        pagination={"pageSize": 5},
        bordered=True,
        sticky=True, # 固定表头
        style={'height': '100%', 'overflowY': 'auto'}
    )

    # 返回结果
    return dict(
        graphs_container=chart,
        table_container=table_container
    )
app.run(port=8090,debug=True)